In [36]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [37]:
url = "https://www.vgchartz.com/games/games.php?console=All"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", response.status_code)
print("Page size:", len(response.text), "characters")
print(response.text[:500])

Status code: 200
Page size: 149649 characters
<!DOCTYPE html>
<html id="XF" lang="en-US" dir="LTR" itemscope>
  <head>
    <title>Video Game Charts, Game Sales, Top Sellers, Game Data - VGChartz</title>

    <base href="https://www.vgchartz.com/" />    
		<meta content="text/html; charset=UTF-8" http-equiv="Content-type"/>
    <meta name="viewport" content="user-scalable=no, width=device-width, initial-scale=1.0" />    <meta name="apple-mobile-web-app-capable" content="yes" />
    <meta http-equiv="X-UA-Compatible" content="IE=Edge" />
    


In [38]:
soup = BeautifulSoup(response.text, "html.parser")

tables = soup.find_all("table")

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print(f"\nTable {i}")
    print(table.get_text(" ", strip=True)[:500])

Number of tables: 8

Table 0
Game Search ( Add new game ) Name: Keyword: Console: - 3DO Interactive Multiplayer Acorn Electron All Amiga Amiga CD32 Amstrad CPC Android Apple II Arcade Atari 2600 Atari 5200 Atari 7800 Atari Jaguar Atari Lynx Atari ST BBC Micro Browser CD-i ColecoVision Commodore 128 Commodore 64 DSi Ware Fairchild Channel F Famicom Disk System FM Towns Game Boy Game Boy Advance Game Boy Color GameCube GameGear Gizmondo HTC Vive Intellivision iOS iQue Linux Mac OS X Magnavox Odyssey Microsoft Windows MS-DOS M

Table 1
Name: Keyword: Console: - 3DO Interactive Multiplayer Acorn Electron All Amiga Amiga CD32 Amstrad CPC Android Apple II Arcade Atari 2600 Atari 5200 Atari 7800 Atari Jaguar Atari Lynx Atari ST BBC Micro Browser CD-i ColecoVision Commodore 128 Commodore 64 DSi Ware Fairchild Channel F Famicom Disk System FM Towns Game Boy Game Boy Advance Game Boy Color GameCube GameGear Gizmondo HTC Vive Intellivision iOS iQue Linux Mac OS X Magnavox Odyssey Microsoft Window

In [39]:
game_table = tables[6]

for row in game_table.find_all("tr")[:3]:
    print([cell.get_text(" ", strip=True)
        for cell in row.find_all(["th", "td"])])

for row in game_table.find_all("tr")[1:6]:
    print([cell.get_text(" ", strip=True)
        for cell in row.find_all(["th", "td"])])

['Results: (1,348)', '<< 1 2 3 4 5 6 >>']
['A B C D E F G H I J K L M N O P Q R S T U V W X Y Z']
['Pos', 'Game', 'Console', 'Publisher', 'VGChartz Score', 'Critic Score', 'User Score', 'Total Shipped', 'Release Date', 'Last Update']
['A B C D E F G H I J K L M N O P Q R S T U V W X Y Z']
['Pos', 'Game', 'Console', 'Publisher', 'VGChartz Score', 'Critic Score', 'User Score', 'Total Shipped', 'Release Date', 'Last Update']
['1', '', 'Minecraft', '', 'Mojang', 'N/A', '8.4', 'N/A', '425.00m', '18th Nov 11', '08th Oct 20']
['2', '', 'Grand Theft Auto V', '', 'Rockstar Games', 'N/A', '9.7', 'N/A', '230.00m', '17th Sep 13', '08th Oct 20']
['3', '', 'Red Dead Redemption 2', '', 'Rockstar Games', 'N/A', '9.8', 'N/A', '87.00m', '26th Oct 18', '08th Oct 20']


In [45]:
rows = []

for tr in game_table.find_all("tr"):
    cells = tr.find_all("td", recursive=False)

    if len(cells) != 11:
        continue

    game_name = cells[2].get_text(" ", strip=True)

    if not game_name:
        continue

    console_img = cells[3].find("img")
    console = console_img.get("alt", "").strip() if console_img else ""


    row = [
        cells[0].get_text(" ", strip=True),  
        game_name,                           
        console,                             
        cells[4].get_text(" ", strip=True),  
        cells[5].get_text(" ", strip=True),  
        cells[6].get_text(" ", strip=True),  
        cells[7].get_text(" ", strip=True),  
        cells[8].get_text(" ", strip=True),  
        cells[9].get_text(" ", strip=True),  
        cells[10].get_text(" ", strip=True)  
    ]

    rows.append(row)

columns = [
    "Pos", "Game", "Console", "Publisher",
    "VGChartz_Score", "Critic_Score", "User_Score",
    "Total_Shipped", "Release_Date", "Last_Update"
]

df = pd.DataFrame(rows, columns=columns)

display(df.head(10))
print("Dataset shape:", df.shape)

,Pos,Game,Console,Publisher,VGChartz_Score,Critic_Score,User_Score,Total_Shipped,Release_Date,Last_Update
0,1,Minecraft,All,Mojang,N/A,8.4,N/A,425.00m,18th Nov 11,08th Oct 20
1,2,Grand Theft Auto V,All,Rockstar Games,N/A,9.7,N/A,230.00m,17th Sep 13,08th Oct 20
2,3,Red Dead Redemption 2,All,Rockstar Games,N/A,9.8,N/A,87.00m,26th Oct 18,08th Oct 20
3,4,Ark: Survival Evolved,All,Studio Wildcard,N/A,N/A,N/A,76.00m,29th Aug 17,31st Dec 20
4,5,PlayerUnknown's Battlegrounds,All,PUBG Corporation,N/A,N/A,N/A,75.00m,17th Dec 17,24th Oct 20
5,6,Terraria,All,Re-Logic,N/A,N/A,N/A,70.00m,16th May 11,10th Oct 20
6,7,The Witcher 3: Wild Hunt,All,Warner Bros. Interactive Entertainment,N/A,8.9,N/A,65.00m,18th May 15,29th Oct 20
7,8,The Elder Scrolls V: Skyrim,All,Bethesda Softworks,N/A,9.4,N/A,65.00m,10th Nov 11,27th Oct 20
8,9,Human: Fall Flat,All,Curve Digital,N/A,N/A,N/A,55.00m,22nd Jul 16,11th Feb 21
9,10,Stardew Valley,All,ConcernedApe,N/A,N/A,N/A,50.00m,26th Feb 16,28th Dec 20


Dataset shape: (50, 10)


In [46]:
df["Release_Date"] = (
    df["Release_Date"]
    .astype("string")
    .str.replace(r"(\d+)(st|nd|rd|th)", r"\1", regex=True)
)

df["Release_Date"] = pd.to_datetime(
    df["Release_Date"],
    format="%d %b %y",
    errors="coerce"
)

df["Year_of_Release"] = df["Release_Date"].dt.year

display(df[["Game", "Release_Date", "Year_of_Release"]].head(10))

print("Valid dates:", df["Release_Date"].notna().sum(), "out of", len(df))

,Game,Release_Date,Year_of_Release
0,Minecraft,2011-11-18,2011
1,Grand Theft Auto V,2013-09-17,2013
2,Red Dead Redemption 2,2018-10-26,2018
3,Ark: Survival Evolved,2017-08-29,2017
4,PlayerUnknown's Battlegrounds,2017-12-17,2017
5,Terraria,2011-05-16,2011
6,The Witcher 3: Wild Hunt,2015-05-18,2015
7,The Elder Scrolls V: Skyrim,2011-11-10,2011
8,Human: Fall Flat,2016-07-22,2016
9,Stardew Valley,2016-02-26,2016


Valid dates: 50 out of 50


In [41]:
print("Number of records:", len(df))
print("\nPlatform values:")
print(df["Console"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Number of records: 50

Platform values:
Console
All    50
Name: count, dtype: int64

Missing values:
Pos               0
Game              0
Console           0
Publisher         0
VGChartz_Score    0
Critic_Score      0
User_Score        0
Total_Shipped     0
Release_Date      0
Last_Update       0
dtype: int64

Data types:
Pos               str
Game              str
Console           str
Publisher         str
VGChartz_Score    str
Critic_Score      str
User_Score        str
Total_Shipped     str
Release_Date      str
Last_Update       str
dtype: object


In [42]:

df["Pos"] = pd.to_numeric(df["Pos"], errors="coerce")

for col in ["VGChartz_Score", "Critic_Score", "User_Score"]:
    df[col] = pd.to_numeric(
        df[col].replace("N/A", pd.NA),
        errors="coerce"
    )

df["Total_Shipped_Millions"] = pd.to_numeric(
    df["Total_Shipped"]
    .str.replace("m", "", regex=False)
    .str.strip(),
    errors="coerce"
)

df["Release_Date"] = pd.to_datetime(
    df["Release_Date"],
    format="%d %b %y",
    errors="coerce"
)

display(df.head())
print(df.dtypes)

,Pos,Game,Console,Publisher,VGChartz_Score,Critic_Score,User_Score,Total_Shipped,Release_Date,Last_Update,Total_Shipped_Millions
0,1,Minecraft,All,Mojang,NaN,8.4,NaN,425.00m,NaT,08th Oct 20,425.0
1,2,Grand Theft Auto V,All,Rockstar Games,NaN,9.7,NaN,230.00m,NaT,08th Oct 20,230.0
2,3,Red Dead Redemption 2,All,Rockstar Games,NaN,9.8,NaN,87.00m,NaT,08th Oct 20,87.0
3,4,Ark: Survival Evolved,All,Studio Wildcard,NaN,NaN,NaN,76.00m,NaT,31st Dec 20,76.0
4,5,PlayerUnknown's Battlegrounds,All,PUBG Corporation,NaN,NaN,NaN,75.00m,NaT,24th Oct 20,75.0


Pos                               int64
Game                                str
Console                             str
Publisher                           str
VGChartz_Score                  float64
Critic_Score                    float64
User_Score                      float64
Total_Shipped                       str
Release_Date              datetime64[s]
Last_Update                         str
Total_Shipped_Millions          float64
dtype: object


In [43]:
df["Release_Date"] = (
    df["Release_Date"]
    .str.replace(
        r"(\d+)(st|nd|rd|th)",
        r"\1",
        regex=True
    )
)

df["Release_Date"] = pd.to_datetime(
    df["Release_Date"],
    format="%d %b %y",
    errors="coerce"
)

df["Year_of_Release"] = df["Release_Date"].dt.year

display(df[["Game", "Release_Date", "Year_of_Release"]].head(10))

print(
    "Valid dates:",
    df["Release_Date"].notna().sum(),
    "out of",
    len(df)
)

AttributeError: Can only use .str accessor with string values, not datetime64

In [44]:
print(df["Release_Date"].dtype)
print(df["Release_Date"].head())

datetime64[s]
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: Release_Date, dtype: datetime64[s]
